# Caso 9 – Segmentación de patrones de compra en una tienda usando clustering

En este caso trabajaremos con una base de datos de órdenes de compra realizadas en una tienda, con el objetivo de identificar diferentes grupos de clientes de acuerdo con sus patrones de compra. Las variables disponibles incluyen el día y la hora de la compra, si el producto fue reordenado, cuántos días pasaron desde la última orden y el número de ítems añadidos al carrito. También aparece información del Departamento y el ítem que se adquirió. Aplicaremos técnicas de reducción de dimensionalidad como el Análisis de Componentes Principales (PCA) para facilitar la visualización, y luego usaremos métodos de clustering como K-means, DBSCAN y Agglomerative Clustering para segmentar a los consumidores según sus patrones de compra. Compararemos el desempeño de los distintos modelos para identificar cuál ofrece una segmentación más útil y coherente.

## Resultado Previsto de Aprendizaje
Al finalizar este caso, el estudiante será capaz de:

- Aplicar técnicas de preprocesamiento y escalamiento para preparar datos numéricos para clustering.
- Implementar reducción de dimensionalidad con PCA para facilitar el análisis visual.
- Utilizar modelos de clustering como K-means, DBSCAN y Agglomerative Clustering usando scikit-learn.

### Contexto del problema
Usted forma parte del equipo de análisis de una cadena de tiendas y busca identificar patrones de compra agrupando órdenes según su comportamiento. El objetivo es poder identificar distintos tipos de clientes para, potencialmente, realizar campañas de marketing diferenciadas.

Iniciemos importando las librerías necesarias

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm

# Cargando y Analizando la Base de datos
Ahora importamos nuestra base de datos

In [ ]:
consumer_data = pd.read_csv('data/Consumer_behavior.csv')
consumer_data

Analicemos ahora la estadística de las variables numéricas

In [ ]:
consumer_data.describe()

Veamos qué tipos de datos tenemos

In [ ]:
consumer_data.dtypes

Ahora miremos cuántos departamentos, productos y usuarios tenemos

In [ ]:
print('Cantidad de departamentos: ', consumer_data['department_id'].nunique())
print('Cantidad de productos: ', consumer_data['product_id'].nunique())
print('Cantidad de usuarios: ', consumer_data['user_id'].nunique())

Verifiquemos si tenemos valores faltantes en nuestra base de datos

In [ ]:
consumer_data.isna().sum()

La columna `days_since_prior_order`, que hace referencia al tiempo entre la última compra y la presente, tiene valores faltantes. Este parámetro identifica las órdenes hechas por primera vez por un cliente. De momento, lo dejaremos de esta manera.

Veamos ahora cómo se distribuye esta variable en nuestra base de datos

In [ ]:
consumer_data['days_since_prior_order'].value_counts().sort_index().plot(kind='bar', figsize=(12, 6))

Como se observa, hay un alto número de compras que se realizan con periodicidad semanal o periodicidad mensual.

## Análisis de compras por día de la semana
Empecemos a analizar nuestra base de datos mirando qué días de la semana se presentan más órdenes, en nuestra base de datos usaremos la columna `order_dow` (day of the week).

In [ ]:
# creando una figura con dos columnas
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# Gráfico del número de ítems pedidos por día de la semana
sns.countplot(x='order_dow', data=consumer_data, ax=axes[0])
# Gráfico del número de órdenes por día de la semana
sns.barplot(x='order_dow', y='order_id', data=consumer_data.groupby('order_dow')['order_id'].nunique().reset_index(), ax=axes[1])
# Agregando títulos a los gráficos
axes[0].set_title('Número de ítems pedidos por día de la semana')
axes[1].set_title('Número de órdenes por día de la semana')
# Agregando etiquetas a los ejes
axes[0].set_xlabel('Día de la semana')
axes[1].set_xlabel('Día de la semana')
axes[0].set_ylabel('Número de ítems')
axes[1].set_ylabel('Número de órdenes')
plt.tight_layout()


Graficando el promedio de ítems pedidos por orden, por día de la semana

In [ ]:
consumer_data.groupby(['order_id','order_dow'])['product_id'].count().reset_index().groupby('order_dow')['product_id'].mean().plot(kind='bar', figsize=(12, 6))
# agregando título y etiquetas
plt.title('Número promedio de productos por orden por día de la semana')
plt.xlabel('Día de la semana')
plt.ylabel('Número promedio de productos')


Ahora analicemos los patrones de las compras del día, en qué momento se presentan más órdenes

In [ ]:
plt.figure(figsize = (7,5))
sns.countplot(data=consumer_data, x=consumer_data['order_hour_of_day'], hue = consumer_data['order_hour_of_day'], legend=False, palette = 'viridis', alpha=0.6)
plt.xlabel("Hora del día", fontsize=12)
plt.ylabel("Número de productos", fontsize=12)
plt.show()

Aquí usamos el tipo de gráfica `countplot` de seaborn, esta gráfica cuenta la cantidad de veces que se repite un dato específico en nuestra base de datos, en este caso, la hora. Como cada producto cuenta como una línea adicional en la base de datos, el resultado es un total sobre el número de productos vendidos en cada franja horaria.

## Análisis de compras por Departamento y por Producto
Veamos cúal es el departamento con más órdenes

In [ ]:
# crando la gráfica mostrando los departamentos más comprados
plt.figure(figsize=(12, 6))
sns.countplot(data=consumer_data, y='department', order=consumer_data['department'].value_counts().index, hue='department', legend = False,  palette='viridis', alpha=0.6)
plt.title('Número de productos comprados por departamento', fontsize=16)
plt.xlabel('Número de productos', fontsize=12)
plt.ylabel('Departamento', fontsize=12)


Y ahora cuáles son los productos más pedidos

In [ ]:
# creando una figura para visualizar los productos más vendidos
plt.figure(figsize=(12, 6))
sns.countplot(data=consumer_data, y='product_name', order=consumer_data['product_name'].value_counts().index[:20], palette='viridis', alpha=0.6)
plt.title('Top 20 productos más vendidos', fontsize=16)
plt.xlabel('Número de productos', fontsize=12)
plt.ylabel('Producto', fontsize=12)

Podemos también observar cuál es el producto más exitoso por cada departamento, para eso usaremos la función `pd.crosstab()`, este nos genera tablas de contingencia, es decir, tablas que muestran la frecuencias conjunta de dos o más variables categóricas

In [ ]:
producto_dept = pd.crosstab(consumer_data['department'], consumer_data['product_name'])
producto_dept

Ahora lo que haremos será identificar por cada fila (Departamento), cuál es la columna con el mayor conteo

In [ ]:
# Nos quedamos solo con el producto más exitoso, es decir, con mayor frecuencia
producto_dept.idxmax(axis=1)

___
#### Ejercicio 1
Use la función `crosstab` para determinar la hora en la que más pedidos se presentan para cada departamento
___

Otra manera de corelacionar variables categóricas es con la función `pd.pivot_table()`, la cual sirve para resumir y reorganizar datos en forma de tabla, agrupando por una o más variables categóricas y aplicando una función de agregación (como suma, promedio, conteo, etc.) sobre columnas numéricas. En este caso, hagamóslo para el departamento y el día de la semana, usando como función de agregación la función `count` aplicada sobre la columna `product_id`. El resultado será el conteo de los productos comprados, por día de la semana, en cada uno de los departamentos.

In [ ]:
dpto_dow = consumer_data.pivot_table(index='department', columns='order_dow', aggfunc='count', values = 'product_id', fill_value=0)
dpto_dow.head(6)

En este punto, nuestra `pivot_table` luce similar a una tabla de contingencia; sin embargo, ofrece una mayor versatilidad al permitir aplicar distintas funciones de agregación. Ahora bien, para analizar si los días de la semana influyen en las ventas por departamento, es necesario normalizar los datos respecto al total de ventas de cada departamento. Esto nos permitirá comparar proporciones en lugar de valores absolutos. Para visualizar este efecto de forma más intuitiva, utilizaremos un heatmap con la librería `seaborn`.

In [ ]:
# normalizamos con respecto a cada fila (Departamento)
dpto_dow =dpto_dow.div(dpto_dow.sum(axis=1), axis =0)
dpto_dow.head(5)

In [ ]:
plt.figure(figsize=(7,8))
sns.heatmap(dpto_dow)
plt.title('Frequency of Orders for Each Department and Hour of Day Combination')
plt.show()

Como se observa, todos los departamentos tienen la misma tendencia con la mayoría de las compras realizadas en el domingo (día 0), o el lunes (día 1). Sin embargo, el alcohol tiene un comportamiento diferente, con la mayoría de los productos comprados el jueves, viernes o sábado.

___
#### Ejercicio 2
Grafique las frecuencias normalizadas en un heatmap de `seaborn` en el que se relacionen las variables de hora del la órden y día de la órden, haga esto usando la función `pivot_table`
___

# Creando una base de datos con Propiedades de Clientes

Nuestro objetivo es identificar clientes con diferentes patrones de compra. Para lograrlo, crearemos una base de datos en la que cada línea es un cliente diferente, y las columnas corresponden a diferentes propiedades de los clientes que pueden ser extraídas de la base de datos.

Iniciemos calculando el total de pedidos realizado por cada cliente:

In [ ]:
clientes_df = consumer_data.groupby('user_id')[['order_id']].nunique()
clientes_df.rename(columns={'order_id': 'num_orders'}, inplace=True)
clientes_df

Agreguemos a esta base de datos, el tiempo promedio entre pedidos para cada cliente. En este cálculo, la primera compra, que aparece con un `nan` en la columna `days_since_prior_order` no será tenida en cuenta en el cálculo. Sin embargo, si un cliente sólo tiene una compra, el cálculo del promedio será `nan`. Para evitar estos valores vacíos y no perder información, llenaremos nuestra base final con -1, y este número identificará los clientes con una única compra en la base de datos.

In [ ]:
clientes_df['avg_time_between_orders'] = consumer_data.groupby('user_id')['days_since_prior_order'].mean().fillna(-1)
clientes_df

Incluyamos ahora el número de productos promedio solicitado por orden

In [ ]:
clientes_df['avg_products_per_order'] = consumer_data.groupby('user_id')['product_id'].count() / clientes_df['num_orders']
clientes_df

# Creando el primer modelo de Clusterización

En este punto podemos crear un primer modelo de clusterización, enfocado en la frecuencia y el tamaño de las órdenes recibidas, de acuerdo con la base de datos.

In [ ]:
# Importamos las librerías
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score
from sklearn.preprocessing import StandardScaler

In [ ]:
plt.figure(figsize=(5,5))
sns.heatmap(clientes_df.corr(numeric_only = True), 
            annot=True,
            vmin=-1,
            vmax=1,
            cmap='coolwarm', 
            fmt='.2f')
plt.tight_layout()
plt.show()

In [ ]:
# scaling the data with standardizing the features 
scalar = StandardScaler()
scaled_data = scalar.fit_transform(clientes_df) # ajusta el escalador a los datos y los transforma en un solo paso.

### Estimando un número ideal de clústers

Un problema natural que surge al intentar ejecutar un algoritmo de segmentación es determinar el número de clústeres (o grupos) en los que se separarán los datos. Este tarea no es evidente por tratarse de datos sin ningún tipos de etiquetas. Sin embargo, existen métodos para estimar cuál podría ser un número adecuado de clústeres para dividir nuestro conjunto de datos.

#### El método del codo

El método del codo (o elbow method en inglés) es una heurística utilizada para determinar el número óptimo de clústeres (k) en un conjunto de datos. Es uno de los métodos más populares y visuales para algoritmos de clustering como K-Means.

La idea principal es ejecutar el algoritmo de clustering (por ejemplo, K-Means) para un rango de valores de k (por ejemplo, de 1 a 10) y, para cada valor, calcular una métrica de rendimiento. Al graficar esta métrica en función de k, la gráfica resultante suele tener la forma de un brazo. El "codo" de ese brazo representa el punto donde agregar más clústeres ya no aporta una mejora significativa, indicando el valor óptimo para k.

##### La Métrica Clave: Inercia (Within-Cluster Sum of Squares)
La métrica más comúnmente utilizada en el método del codo es la inercia, también conocida como Suma de las distancias al cuadrado dentro de los clústeres (WCSS - Within-Cluster Sum of Squares).

La inercia mide cuán compacto es un clúster. Específicamente, calcula la suma de las distancias al cuadrado de cada punto de datos a su centroide más cercano.

- Una inercia baja significa que los puntos de datos dentro de un clúster están muy juntos y cerca de su centroide, lo que indica clústeres densos y bien definidos.
- Una inercia alta significa que los puntos están más dispersos dentro del clúster.

¿Cómo se Calcula la Inercia?
Matemáticamente, si tenemos un conjunto de datos $X=x_1, x_2, \dots, x_n$,  y un conjunto de clústeres $C=C_1, C_2, \dots, C_k$ con sus respectivos centroides $\mu_j$, la inercia se define como:

$$I=\sum_{j=1}^k\sum_{x_i\in C_j}|x_i-\mu_j|^2$$
 

In [ ]:
# implementando el método del codo para determinar el número óptimo de clusters
inertia = []
for i in tqdm(range(1, 11)):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(scaled_data)
    inertia.append(kmeans.inertia_)
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia, marker='o')
plt.title('Método del Codo para Determinar el Número Óptimo de Clusters')
plt.xlabel('Número de Clusters')
plt.ylabel('Inercia')

Al analizar la gráfica observamos que por encima de $k=4$, ingresar clústeres adicionales no reduce significativamente la inercia. De manera que este método nos sugiere que este valor podría ser un valor adecuado para el número de clústeres.

### El Índice de Calinski-Harabasz para Evaluar Clústeres
El Índice de Calinski-Harabasz (también conocido como Criterio de Razón de Varianza o VRC) es una métrica utilizada para evaluar la calidad de una solución de clustering. A diferencia del método del codo que se basa en la inspección visual de una curva, este índice proporciona una puntuación numérica que ayuda a determinar el número óptimo de clústeres (k) de una forma más objetiva.

La idea central es que un buen clustering tiene clústeres muy compactos (baja dispersión dentro de los clústeres) y, al mismo tiempo, muy bien separados entre sí (alta dispersión entre los clústeres). El índice de Calinski-Harabasz cuantifica esta relación.

#### La Métrica: Razón de Varianzas
El índice se define como la razón entre la dispersión entre los clústeres y la dispersión dentro de los clústeres.

- Una puntuación alta indica que los clústeres son densos y están bien separados. Este es el resultado deseado.
- Una puntuación baja puede indicar que los clústeres no son densos, no están bien separados, o ambas cosas.

¿Cómo se Calcula el Índice?
El cálculo se basa en dos componentes:

- Dispersión entre clústeres (SSB - Sum of Squares Between clusters): Mide qué tan separados están los centroides de los clústeres del centroide global de los datos. Una mayor dispersión entre clústeres es mejor.
- Dispersión dentro de los clústeres (SSW - Sum of Squares Within clusters): Es la suma de las distancias al cuadrado de cada punto a su propio centroide. Es exactamente la misma métrica que la inercia utilizada en el método del codo. Una menor dispersión dentro de los clústeres es mejor.

Matemáticamente, el índice de Calinski-Harabasz (CH) para k clústeres en un conjunto de datos con N muestras se define como:

$$CH(k)= \frac{SSB}{SSW}\times\frac{N-k}{k-1}$$

Donde:

- $SSB=\sum_{j=1}^kn_j|\mu_j-\mu|^2$
- $SSW=\sum_{j=1}^k\sum_{x_i\in C_j}|x_i-\mu_j|^2$
- $\mu$ es el centroide global de todos los datos.
- El término de ajuste $\frac{N-k}{k-1}$ penaliza la creación de más clústeres, ayudando a evitar el sobreajuste.

In [ ]:
# calculando el índice Calinski-Harabasz para cada número de clusters
CH_scores = []
for i in tqdm(range(2, 11)):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(scaled_data)
    score = calinski_harabasz_score(scaled_data, kmeans.labels_)
    CH_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), CH_scores, marker='o')
plt.title('Índice Calinski-Harabasz para Determinar el Número Óptimo de Clusters')
plt.xlabel('Número de Clusters')
plt.ylabel('Índice Calinski-Harabasz')


Como se observa, el índice de Calinski-Harabasz se maximiza precisamente cuando usamos $k=4$ clústeres, lo que confirma que este es un buen punto de partida para nuestro proceso de segmentación de clientes

### Ejecutando k-Means con k=4
Implementemos ahora el algoritmo y analicemos sus resultados

In [ ]:
# Implementando KMeans con el número óptimo de clusters
kmeans = KMeans(n_clusters=4, n_init='auto', random_state=42)
kmeans.fit(scaled_data)
kmeans_labels = kmeans.fit_predict(scaled_data)

Veamos la distribución de cada una de nuestras variables para cada uno de los clústers

In [ ]:
# graficando la distribución del nùmero de órdenes por usuario para cada cluster
plt.figure(figsize=(10, 6))
sns.boxplot(x=kmeans_labels, y = clientes_df['num_orders'], hue = kmeans_labels)
plt.xlabel('clúster')
plt.ylabel('número de órdenes')

Como se observa, el clúster 2 corresponde a los clientes que más ordenes tienen en la base de datos. Para los demás clústeres no se observa una diferencia evidente en el número de órdenes.

In [ ]:
# graficando la distribución del tiempo entre órdenes promedio para cada cluster
plt.figure(figsize=(10, 6))
sns.boxplot(x=kmeans_labels, y = clientes_df['avg_time_between_orders'], hue = kmeans_labels)

Esta gráfica nos permite ver que el clúster 1 contiene principalmente a los clientes que hacen sus órdenes con periodicidad mensual. Para el Clúster 0 hay una periodicidad cercana al 10. Mientras que los clústers 1 están más cercanos a una periodicidad semanal.

In [ ]:
# graficando la distribución del número promedio de productos por compra
plt.figure(figsize=(10, 6))
sns.boxplot(x=kmeans_labels, y = clientes_df['avg_products_per_order'], hue = kmeans_labels)

Esta gráfica nos muestra que al clúster 0 pertenecen principalmente los usuarios que ordenan más productos por compra.

In [ ]:
import plotly.graph_objs as go

fig = go.Figure(data=[go.Scatter3d(
    x=clientes_df.iloc[:,0],
    y=clientes_df.iloc[:,1],
    z=clientes_df.iloc[:,2],
    mode='markers',
    marker=dict(
        size=5,
        color=kmeans_labels, # color por cluster
        colorscale='Viridis',
        opacity=0.8
    )
)])

fig.update_layout(
    title='Visualización 3D de los clusters (Plotly)',
    scene=dict(
        xaxis_title="número de órdenes",
        yaxis_title="tiempo promedio entre órdenes",
        zaxis_title="número de productos por orden"
    ),
    width=800,
    height=800,
)

fig.show()

El algoritmo ha creado finalmente cuatro grupos de clientes:

- Grupo 1 (verde): los clientes que más ordenes hacen.
- Grupo 2 (morado): los clientes que más productos piden por orden.
- Grupo 3 (amarillo): los clientes que hicieron pocas órdenes pero que las hicieron muy seguidas.
- Grupo 4 (azul): los clientes que hicieron pocas órdenes pero que las espaciaron más.

# Segundo modelo, añadiendo información sobre el día de la semana de compras

Usemos de nuevo la función `pivot_table` para agregar a nuestra base de datos información sobre la distribución de las compras en los días de la semana. La idea es crear una columna para tres franjas posibles, establecidas de acuerdo con el comportamiento por departamentos que vimos anteriormente, en la que aparezca la fracción de órdenes realizada en promedio en cada franja.

In [ ]:
# creando una nueva columna en la base de datos original con las franjas semanales
franja_semanal = ['lun_mar_mie', 'jue_vie', 'sab_dom']
valores_dia = [0, 3, 5, 7]
consumer_data['franja_semanal'] = pd.cut(
    consumer_data['order_dow'],
    bins=valores_dia, 
    labels=franja_semanal,
    right=False,
    ordered=False
)

consumer_data

In [ ]:
consumer_data['franja_semanal'].value_counts()

Usando esta nueva columna podemos crear una nueva base de datos usando la función `pivot_table`

In [ ]:

# calculando la cantidad de órdenes por día de la semana
dow_data = consumer_data.pivot_table(index='user_id', columns='franja_semanal', values='order_id', aggfunc='nunique').fillna(0)
# normalizando con respecto a cada fila (usuario)
dow_data = dow_data.div(dow_data.sum(axis=1), axis=0)
# agregando a la base de datos de clientes
clientes_df = pd.concat([clientes_df, dow_data], axis=1)
clientes_df


In [ ]:
# scaling the data with standardizing the features 
scalar = StandardScaler()
scaled_data = scalar.fit_transform(clientes_df) # ajusta el escalador a los datos y los transforma en un solo paso.

In [ ]:
# implementando el método del codo para determinar el número óptimo de clusters
inertia = []
for i in tqdm(range(1, 11)):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(scaled_data)
    inertia.append(kmeans.inertia_)
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia, marker='o')
plt.title('Método del Codo para Determinar el Número Óptimo de Clusters')
plt.xlabel('Número de Clusters')
plt.ylabel('Inercia')

In [ ]:
# calculando el índice Calinski-Harabasz para cada número de clusters
CH_scores = []
for i in tqdm(range(2, 11)):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(scaled_data)
    score = calinski_harabasz_score(scaled_data, kmeans.labels_)
    CH_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), CH_scores, marker='o')
plt.title('Índice Calinski-Harabasz para Determinar el Número Óptimo de Clusters')
plt.xlabel('Número de Clusters')
plt.ylabel('Índice Calinski-Harabasz')

De nuevo, nos aparece que la separación óptima es de 4 grupos

In [ ]:
# Implementando KMeans con el número óptimo de clusters
kmeans = KMeans(n_clusters=4, n_init='auto', random_state=42)
kmeans.fit(scaled_data)
kmeans_labels = kmeans.fit_predict(scaled_data)

Analicemos cómo se distribuyen los clústeres que se encontraron en la base de datos

In [ ]:
pd.Series(kmeans_labels).value_counts()

Veamos ahora cómo se distribuyen los grupos en las 3 variables originales

In [ ]:
import plotly.graph_objs as go

fig = go.Figure(data=[go.Scatter3d(
    x=clientes_df.iloc[:,0],
    y=clientes_df.iloc[:,1],
    z=clientes_df.iloc[:,2],
    mode='markers',
    marker=dict(
        size=5,
        color=kmeans_labels, # color por cluster
        colorscale='Viridis',
        opacity=0.8,
        colorbar=dict(
            title='Cluster'  # Título de la leyenda
        )
    ),
    showlegend=False,  # No aplica para Scatter3d, pero puedes dejarlo
)])

fig.update_layout(
    title='Visualización 3D de los clusters (Plotly)',
    scene=dict(
        xaxis_title="número de órdenes",
        yaxis_title="tiempo promedio entre órdenes",
        zaxis_title="número de productos por orden"
    ),
    width=800,
    height=800,
)

fig.show()

Como se observa, sólo el Clúster 1 puede ser interpretado como el correspondiente a los clientes que más órdenes generan. Los demás clústeres se encuentran mezclados para todos los clientes con un número de órdenes pequeño. Veamos ahora cómo se distribuyen las proporciones de compras en las distintas franjas semanales para determinar la composición de los clústeres.

In [ ]:
# Suponiendo que kmeans_labels es un array con la etiqueta de cluster para cada usuario
clientes_df['cluster'] = kmeans_labels

# Selecciona solo las columnas de interés y el cluster
df_long = clientes_df[['cluster', 'lun_mar_mie', 'jue_vie', 'sab_dom']].reset_index()
df_long = df_long.melt(id_vars=['cluster', 'user_id'], 
                       value_vars=['lun_mar_mie', 'jue_vie', 'sab_dom'],
                       var_name='Franja', value_name='Proporcion')

# Graficar: un gráfico por cluster
g = sns.catplot(
    data=df_long,
    x='Franja',
    y='Proporcion',
    col='cluster',
    col_wrap=2,
    kind='violin',
    height=5,
    sharey=True
)
g.set_titles("Cluster {col_name}")
g.set_axis_labels("Franja semanal", "Proporción de órdenes")
plt.show()

## Conclusión
El proceso de clústering usando variables de frecuencia y tamaño de pedidos, así como de distribución de pedidos por día de la semana, nos arroja las siguientes configuraciones de grupos:

- Cluster 0: corresponde a los clientes que compran exclusivamente al inicio de la semana.
- Cluster 1: correspondiente a los clientes que más pedidos hicieron a la tienda.
- Cluster 2: correspondiente a los clientes que compran principalmente los jueves o viernes.
- Cluster 3: correspondiente a los clientes que compran principalmente los días sábado y domingo.

Este resultado se debe a la muy fuerte segmentación que existe en la distribución de compras en la semana que existe en la base de datos de clientes.

# Clusterización con DB-SCAN

Usemos ahora un algoritmo diferente para hacer la segmentación de clientes. En este caso, usaremos DB-SCAN un algoritmo que no usa clústeres esféricos, sino que usa la densidad de puntos para establecer los puntos más cercanos entre sí y así formar los clústeres. La principal ventaja de este algoritmo es que no necesita especificar el número de clústeres. Sin embargo, su funcionamiento depende de dos parámetros: `eps` y `min_samples`, que determinan, respectivamente, la distancia máxima entre dos puntos para ser considerados vecinos y el mínimo de puntos en la vecindad de un punto específico para que este sea considerado un punto central.

- Para `min_samples` usaremos $n_{\text{features}}+1=7$
- Escanearemos el valor de `eps` para obtener un número de clústeres que tenga sentido para nosotros.

In [ ]:
# implementando el algoritmo DBSCAN para clusterizar los datos
from sklearn.cluster import DBSCAN
epss = [0.5,0.6,0.7,0.8,0.9,1.0]
for eps in epss:
    # creando un objeto DBSCAN
    dbscan = DBSCAN(eps=eps, min_samples=7)
    # ajustando el modelo a los datos escalados
    dbscan.fit(scaled_data)
    # obteniendo las etiquetas de los clusters
    dbscan_labels = dbscan.labels_
    # imprimiendo el número de clusters encontrados
    n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    print(f'Epsilon: {eps}, Número de clusters: {n_clusters}')


Como se observa, usando un `eps=0.9` obtenemos 4 clústeres, que es el mismo número que habíamos obtenido anteriormente. Este número es ideal puesto que permite hacer un análisis de los clústeres obtenidos y relacionarlos con características específicas de los compradores.

In [ ]:
# creando un objeto DBSCAN
dbscan = DBSCAN(eps=0.9, min_samples=7)
# ajustando el modelo a los datos escalados
dbscan.fit(scaled_data)
# obteniendo las etiquetas de los clusters
dbscan_labels = dbscan.labels_



Veamos cuáles son los labels que crea el algoritmo y cómo se distribuyen los clientes en estos puntos:

In [ ]:
print('Los labels creados', np.unique(dbscan_labels))
print('Distribución de los labels', pd.Series(dbscan_labels).value_counts())

Como se observa, realmente hay 5 categorías, los puntos correspondientes a la categoría -1 son puntos que no pertenecen a ningún clúster y que pueden ser interpretados como anomalías.

Veamos la distribución de los clústeres sobre las variables originales

In [ ]:
import plotly.graph_objs as go

fig = go.Figure(data=[go.Scatter3d(
    x=clientes_df.iloc[:,0],
    y=clientes_df.iloc[:,1],
    z=clientes_df.iloc[:,2],
    mode='markers',
    marker=dict(
        size=5,
        color=dbscan_labels, # color por cluster
        colorscale='Viridis',
        opacity=0.8,
        colorbar=dict(
            title='Cluster'  # Título de la leyenda
        )
    ),
    showlegend=False,  # No aplica para Scatter3d, pero puedes dejarlo
)])

fig.update_layout(
    title='Visualización 3D de los clusters (Plotly)',
    scene=dict(
        xaxis_title="número de órdenes",
        yaxis_title="tiempo promedio entre órdenes",
        zaxis_title="número de productos por orden"
    ),
    width=800,
    height=800,
)

fig.show()

Como se observa, el algoritmo cataloga la gran mayoría de clientes dentro del Clúster 1, los puntos extremos han sido asignados al clúster -1. Los clústeres 0, 2 y 3 parecen estar limitados a únicamente los clientes con exactamente 2 órdenes en la base de datos.

Veamos la distribución de las variables restantes en cada uno de los clústeres.

In [ ]:
# Suponiendo que kmeans_labels es un array con la etiqueta de cluster para cada usuario
clientes_df['cluster'] = dbscan_labels

# Selecciona solo las columnas de interés y el cluster
df_long = clientes_df[['cluster', 'lun_mar_mie', 'jue_vie', 'sab_dom']].reset_index()
df_long = df_long.melt(id_vars=['cluster', 'user_id'], 
                       value_vars=['lun_mar_mie', 'jue_vie', 'sab_dom'],
                       var_name='Franja', value_name='Proporcion')

# Graficar: un gráfico por cluster
g = sns.catplot(
    data=df_long,
    x='Franja',
    y='Proporcion',
    kind='box',
    col='cluster',
    col_wrap=2,
    height=5,
    sharey=True
)
g.set_titles("Cluster {col_name}")
g.set_axis_labels("Franja semanal", "Proporción de órdenes")
plt.show()

## Conclusión

- El método DB Scan crea 4 clústeres más un grupo de puntos anómalos.
- El clúster 1 es el más grande y abarca la mayoría de los puntos.
- El clúster 0 corresponde a clientes con 2 compras, una `lun_mar_mie` y una en `sab_dom`.
- El clúster 2 corresponde a clientes con 2 compras, una en `jue_vie` y una en `sab_dom`.
- El clúster 3 corresponde a clientes con 2 compras, una en `lun_mar_mie` y una en `jue_vie`.


Esta clusterización es diferente a la propuesta por el método de K-Means, desde un punto de vista de negocio puede no ser tan útil debido a que los clústers no son representativos de diferentes patrones de compra. Como sí ocurrió en el caso de K-Means.

# Aplicando PCA para reducir y cambiar las variables
El análisis de componentes principales permite crear un nuevo conjunto de variables que son combinaciones lineales de las variables originales. Este método es muy útil para hacer reducciones de dimensionalidad, pero también es muy útil para transformar el espacio de datos y explorar nuevas posibilidades de clusterización.

Para ver con cuántos componentes vamos a trabajar, podemos revisar cuál es la varianza explicada en función del número de componentes prinicpales, esto nos permite establecer con cuántos componentes trabajar

In [ ]:
# calculando la varianza acomulada explicada en función del número de componentes principales
pca = PCA()
pca.fit(scaled_data)
# graficando la varianza acumulada explicada
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_.cumsum(), marker='o')
plt.title('Varianza Acumulada Explicada por los Componentes Principales')
plt.xlabel('Número de Componentes Principales')
plt.ylabel('Varianza Acumulada Explicada')


Como se observa, en este caso, con 3 componentes tenemos alrededor del 70% de la varianza explicada. Transformaremos entonces nuestra base de datos usando 3 componentes principales

In [ ]:
# transformando los datos originales a las componentes principales
pca = PCA(n_components=3)
pca_data = pca.fit_transform(scaled_data)

Hagamos de nuevo K-Means con las 3 componentes principales seleccionadas

In [ ]:
# haciendo kmeans con las componentes principales
kmeans = KMeans(n_clusters=4, n_init='auto', random_state=42)
kmeans.fit(pca_data)
kmeans_labels = kmeans.fit_predict(pca_data)

Observemos cómo se ven los clústeres en el espacio de 3D de componentes principales

In [ ]:
fig = go.Figure(data=[go.Scatter3d(
    x=pca_data[:,0],
    y=pca_data[:,1],
    z=pca_data[:,2],
    mode='markers',
    marker=dict(
        size=5,
        color=kmeans_labels, # color por cluster
        colorscale='Viridis',
        opacity=0.8,
        colorbar=dict(
            title='Cluster'  # Título de la leyenda
        )
    ),
    showlegend=False,  # No aplica para Scatter3d, pero puedes dejarlo
)])

fig.update_layout(
    title='Visualización 3D de los clusters, en el espacio de componentes principales (Plotly)',
    scene=dict(
        xaxis_title="pc1",
        yaxis_title="pc2",
        zaxis_title="pc3"
    ),
    width=800,
    height=800,
)

fig.show()

Ahora veamos cómo se ve el resultado en el espacio original de variables

In [ ]:
fig = go.Figure(data=[go.Scatter3d(
    x=clientes_df.iloc[:,0],
    y=clientes_df.iloc[:,1],
    z=clientes_df.iloc[:,2],
    mode='markers',
    marker=dict(
        size=5,
        color=kmeans_labels, # color por cluster
        colorscale='Viridis',
        opacity=0.8,
        colorbar=dict(
            title='Cluster'  # Título de la leyenda
        )
    ),
    showlegend=False,  # No aplica para Scatter3d, pero puedes dejarlo
)])

fig.update_layout(
    title='Visualización 3D de los clusters (Plotly)',
    scene=dict(
        xaxis_title="número de órdenes",
        yaxis_title="tiempo promedio entre órdenes",
        zaxis_title="número de productos por orden"
    ),
    width=800,
    height=800,
)

fig.show()

Y para las variables relacionadas con la franja de la semana en la que se hacen las compras

In [ ]:
# Suponiendo que kmeans_labels es un array con la etiqueta de cluster para cada usuario
clientes_df['cluster'] = kmeans_labels

# Selecciona solo las columnas de interés y el cluster
df_long = clientes_df[['cluster', 'lun_mar_mie', 'jue_vie', 'sab_dom']].reset_index()
df_long = df_long.melt(id_vars=['cluster', 'user_id'], 
                       value_vars=['lun_mar_mie', 'jue_vie', 'sab_dom'],
                       var_name='Franja', value_name='Proporcion')

# Graficar: un gráfico por cluster
g = sns.catplot(
    data=df_long,
    x='Franja',
    y='Proporcion',
    kind='violin',
    col='cluster',
    col_wrap=2,
    height=5,
    sharey=True
)
g.set_titles("Cluster {col_name}")
g.set_axis_labels("Franja semanal", "Proporción de órdenes")
plt.show()

## Conclusión

El uso de sólo 3 componentes principales permite realizar el mismo proceso de segmentación de clientes que el uso de las variables originales, esto muestra el poder de PCA y su capacidad de reducir tiempos de cómputo y complejidad de visuaización.

___
#### Ejercicio 3
Partiendo únicamente de las 3 variables relacionadas con el número, la frecuencia y la cantidad de productos por orden, use PCA para reducir las variables a 1 o 2 componentes principales y repita el algoritmo de clusterización con K-Means sobre las componentes principales. ¿Cómo cambia el resultado de la clusterización? ¿Se reproducen los resultados obtenidos al inicio del caso?
___

# Conclusión General

Los algoritmos de clusterización permiten separar los clientes de acuerdo con sus caracyerísticas de compra. El resultado del proceso de separación depende tanto de las variables que se incluyan en el modelo como del algoritmo que se utilice para hacer la segmentación. La selección de variables y del algoritmo dependerá de los requerimientos del proceso y de la evaluación de los resultados por parte de un experto

### Origen de los datos

Ventas supermercado https://www.kaggle.com/datasets/hunter0007/ecommerce-dataset-for-predictive-marketing-2023/code 